# QNG Optimizer — Interactive Demo

**Paper**: *Quantum Natural Gradient Optimization for Convergence Reliability in NISQ Variational Quantum Algorithms*  
**Author**: Mezbah Uddin Rafi, Council for International Artificial Intelligence Union, Geneva

This notebook walks through the full pipeline:
1. Build the 4-qubit MaxCut problem
2. Construct the hardware-efficient ansatz
3. Apply a NISQ noise model
4. Run Vanilla Gradient Descent vs QNG
5. Visualize results

**Runtime**: ~5–15 minutes for 5 trials (adjust `N_TRIALS` and `N_SHOTS` below).

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))   # Add repo root to path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 110})

from src.ansatz import build_ansatz, get_parameter_count
from src.cost_function import MaxCutCostFunction
from src.noise_model import build_noise_model, print_noise_summary
from src.qng_optimizer import QNGOptimizer
from src.vanilla_gd import VanillaGradientDescent

print('Imports OK')

## 1. Configure the Experiment

Adjust these parameters to trade speed for accuracy.  
For exact paper replication: `N_TRIALS=50`, `N_SHOTS=8192`, `QFIM_MODE='full'`.

In [ ]:
# ---- Experiment settings ----
N_QUBITS   = 4        # Number of qubits (paper: 4)
N_LAYERS   = 3        # Ansatz layers (paper: 3)
N_TRIALS   = 5        # Trials per optimizer (paper: 50) — reduce for speed
N_SHOTS    = 1024     # Shots per eval (paper: 8192) — reduce for speed
PLATFORM   = 'trapped_ion'   # 'trapped_ion', 'superconducting', 'high_noise'
MAX_ITER   = 30       # Max iterations (paper: 100)
QFIM_MODE  = 'diagonal'  # 'diagonal' (fast) or 'full' (paper-accurate)
BASE_SEED  = 42

N_PARAMS = get_parameter_count(N_QUBITS, N_LAYERS)
print(f'Parameters: {N_PARAMS}  (= {N_QUBITS} qubits × 2 rotations × {N_LAYERS} layers)')

## 2. Build Problem Components

In [ ]:
# ---- 2a. Ansatz circuit ----
circuit = build_ansatz(n_qubits=N_QUBITS, n_layers=N_LAYERS)
print('Ansatz circuit:')
print(circuit.draw(output='text', fold=80))

In [ ]:
# ---- 2b. MaxCut cost function (paper's graph: Section II.B) ----
EDGE_WEIGHTS = {(0,1): 1.5, (0,2): 1.0, (1,3): 0.8, (2,3): 1.2}
cost_fn = MaxCutCostFunction(n_qubits=N_QUBITS, edge_weights=EDGE_WEIGHTS)

print(f'MaxCut problem:')
print(f'  Edges: {EDGE_WEIGHTS}')
print(f'  Optimal cost (classical): {cost_fn.optimal_value}')
print(f'\nHamiltonian (SparsePauliOp):')
print(cost_fn.get_hamiltonian())

In [ ]:
# ---- 2c. NISQ noise model ----
noise_model = build_noise_model(platform=PLATFORM)
print_noise_summary(PLATFORM)

## 3. Run Optimization Trials

In [ ]:
from src.qng_optimizer import OptimizationResult
from typing import List

def run_trials(optimizer_class, optimizer_kwargs, n_trials, base_seed):
    """Run n_trials and return list of OptimizationResult."""
    results = []
    for i in range(n_trials):
        seed = base_seed + i
        np.random.seed(seed)
        theta_init = np.random.uniform(-0.1, 0.1, N_PARAMS)
        optimizer = optimizer_class(**optimizer_kwargs)
        result = optimizer.optimize(
            theta_init=theta_init,
            cost_fn=cost_fn,
            circuit=circuit,
            noise_model=noise_model,
            seed=seed,
        )
        print(f'  Trial {i+1}/{n_trials}: converged={result.converged}, '
              f'iters={result.iterations}, cost={result.final_cost:.3f}')
        results.append(result)
    return results

common_kwargs = dict(
    learning_rate=0.1,
    max_iterations=MAX_ITER,
    convergence_threshold=0.1,
    n_shots=N_SHOTS,
)

print('=== Vanilla Gradient Descent ===')
vgd_results = run_trials(VanillaGradientDescent, common_kwargs, N_TRIALS, BASE_SEED)

print('\n=== Quantum Natural Gradient ===')
qng_kwargs = {**common_kwargs, 'regularization': 1e-4, 'qfim_mode': QFIM_MODE}
qng_results = run_trials(QNGOptimizer, qng_kwargs, N_TRIALS, BASE_SEED)

## 4. Summary Statistics

In [ ]:
def summarize(results: List[OptimizationResult], label: str):
    converged = [r.converged for r in results]
    iters = [r.iterations for r in results]
    conv_iters = [r.iterations for r in results if r.converged]
    times = [r.wall_clock_time for r in results]
    costs = [r.final_cost for r in results]
    
    print(f'\n{label}:')
    print(f'  Convergence rate : {np.mean(converged)*100:.0f}% ({sum(converged)}/{len(converged)})')
    if conv_iters:
        print(f'  Mean iters (conv): {np.mean(conv_iters):.1f} ± {np.std(conv_iters):.1f}')
    print(f'  Wall-clock time  : {np.mean(times):.1f} ± {np.std(times):.1f} s')
    print(f'  Final cost       : {np.mean(costs):.3f} ± {np.std(costs):.3f}')

print('=== Experiment Summary ===')
print(f'Platform: {PLATFORM} | Trials: {N_TRIALS} | Max iter: {MAX_ITER}')
summarize(vgd_results, 'Vanilla GD')
summarize(qng_results, 'QNG')

# Improvement ratio
vgd_sr = np.mean([r.converged for r in vgd_results])
qng_sr = np.mean([r.converged for r in qng_results])
if vgd_sr > 0:
    print(f'\nConvergence improvement: {qng_sr/vgd_sr:.2f}×  (paper: 3.2×)')

## 5. Visualizations

In [ ]:
def pad_histories(results, key='cost_history'):
    histories = [getattr(r, key) for r in results if getattr(r, key)]
    max_len = max(len(h) for h in histories)
    return np.array([h + [h[-1]] * (max_len - len(h)) for h in histories])

COLORS = {'VGD': '#E84855', 'QNG': '#2E86AB'}

# ---- Plot 1: Convergence curves ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for label, results, color in [
    ('Vanilla GD', vgd_results, COLORS['VGD']),
    ('QNG', qng_results, COLORS['QNG']),
]:
    matrix = pad_histories(results, 'cost_history')
    mean = np.mean(matrix, axis=0)
    std = np.std(matrix, axis=0)
    xs = np.arange(len(mean))
    ax.plot(xs, mean, color=color, label=label, linewidth=2.2)
    ax.fill_between(xs, mean-std, mean+std, alpha=0.15, color=color)

ax.axhline(y=cost_fn.optimal_value, color='gray', linestyle='--', alpha=0.7, label='Optimal')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost ⟨H⟩')
ax.set_title(f'Convergence Curves ({N_TRIALS} trials, ±1 std)')
ax.legend()

# ---- Plot 2: Success rates ----
ax2 = axes[1]
vgd_sr = np.mean([r.converged for r in vgd_results]) * 100
qng_sr = np.mean([r.converged for r in qng_results]) * 100

bars = ax2.bar(['Vanilla GD', 'QNG'], [vgd_sr, qng_sr],
               color=[COLORS['VGD'], COLORS['QNG']], edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, [vgd_sr, qng_sr]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{rate:.0f}%', ha='center', fontweight='bold', fontsize=13)

ax2.set_ylim(0, 115)
ax2.set_ylabel('Convergence Success Rate (%)')
ax2.set_title(f'Success Rate\n({PLATFORM} noise)')
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.3)

fig.suptitle('QNG vs Vanilla GD — Key Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3: Gradient norm evolution ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for label, results, color in [
    ('Vanilla GD', vgd_results, COLORS['VGD']),
    ('QNG', qng_results, COLORS['QNG']),
]:
    matrix = pad_histories(results, 'gradient_norm_history')
    if matrix.size == 0:
        continue
    mean = np.mean(matrix, axis=0)
    std = np.std(matrix, axis=0)
    xs = np.arange(len(mean))
    ax.semilogy(xs, mean + 1e-10, color=color, label=label, linewidth=2.0)
    ax.fill_between(xs, np.maximum(mean-std, 1e-10), mean+std, alpha=0.15, color=color)

ax.set_xlabel('Iteration')
ax.set_ylabel('Gradient Norm ‖∇L‖ (log scale)')
ax.set_title('Gradient Norm Over Training\nQNG resists barren plateau vanishing')
ax.legend()

# ---- Plot 4: Final cost distribution ----
ax2 = axes[1]
vgd_costs = [r.final_cost for r in vgd_results]
qng_costs = [r.final_cost for r in qng_results]

bp = ax2.boxplot([vgd_costs, qng_costs], patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], [COLORS['VGD'], COLORS['QNG']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.axhline(y=cost_fn.optimal_value, color='gray', linestyle='--', alpha=0.7, label='Optimal')
ax2.set_xticklabels(['Vanilla GD', 'QNG'])
ax2.set_ylabel('Final Cost ⟨H⟩')
ax2.set_title('Final Solution Quality')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Single-Trial Deep Dive

Inspect the best QNG trial in detail.

In [ ]:
# Find best QNG trial
best_qng = min(qng_results, key=lambda r: r.final_cost)
print(f'Best QNG trial (seed={best_qng.seed}):')
print(f'  Converged  : {best_qng.converged}')
print(f'  Iterations : {best_qng.iterations}')
print(f'  Final cost : {best_qng.final_cost:.4f}  (optimal: {cost_fn.optimal_value})')
print(f'  Time       : {best_qng.wall_clock_time:.1f} s')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(best_qng.cost_history, 'o-', color=COLORS['QNG'], markersize=5, linewidth=2)
ax.axhline(y=cost_fn.optimal_value, color='gray', linestyle='--', alpha=0.7, label='Optimal')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost ⟨H⟩')
ax.set_title(f'Best QNG Trial — Single-Trial Convergence (seed={best_qng.seed})')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Barren Plateau Illustration

Synthetic visualization of gradient variance decay with qubit count.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

qubit_counts = np.arange(2, 16)

# Theoretical gradient variance decay
var_global = 2.0 ** (-qubit_counts.astype(float))        # Global cost: exponential
var_local  = qubit_counts.astype(float) ** (-2)          # Local cost: polynomial

# Normalize to 1 at n=2
ax.semilogy(qubit_counts, var_global / var_global[0], 'o-',
            color=COLORS['VGD'], linewidth=2, markersize=6, label='Global cost (exponential)')
ax.semilogy(qubit_counts, var_local / var_local[0], 's--',
            color=COLORS['QNG'], linewidth=2, markersize=6, label='Local cost (polynomial)')

ax.axvline(x=10, color='black', linestyle=':', alpha=0.5)
ax.text(10.2, 0.1, 'n=10: global variance\n~10⁻¹² of n=2 value', fontsize=9)

ax.set_xlabel('Number of Qubits n', fontsize=12)
ax.set_ylabel('Normalized Gradient Variance', fontsize=12)
ax.set_title('Barren Plateau: Gradient Variance Decay vs System Size\n'
             '(McClean et al. 2018; Cerezo et al. 2021)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(qubit_counts)
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
import json
from datetime import datetime

notebook_results = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'n_qubits': N_QUBITS, 'n_layers': N_LAYERS,
        'n_trials': N_TRIALS, 'n_shots': N_SHOTS,
        'platform': PLATFORM, 'max_iterations': MAX_ITER,
    },
    'QNG': [r.to_dict() for r in qng_results],
    'VGD': [r.to_dict() for r in vgd_results],
}

os.makedirs('../results', exist_ok=True)
out_path = f'../results/notebook_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(out_path, 'w') as f:
    json.dump(notebook_results, f, indent=2)
    
print(f'Results saved to: {out_path}')

## 9. Next Steps

- **Full reproduction**: Set `N_TRIALS=50`, `N_SHOTS=8192`, `QFIM_MODE='full'`
- **Run CLI**: `python experiments/run_experiments.py --n_trials 50 --platform trapped_ion`
- **Other noise models**: Change `PLATFORM` to `'superconducting'` or `'high_noise'`
- **Read theory**: See `docs/theory.md` for mathematical background
- **Extend**: Add your own optimizer in `src/` following the `BaseCostFunction` interface

**Paper key numbers to reproduce:**

| Metric | Paper | You should see (≈) |
|--------|-------|--------------------|
| QNG success rate | 95% | ~90–95% |
| VGD success rate | 30% | ~25–40% |
| QNG mean iters | 15 | ~12–20 |
| QNG final cost | -3.10 | ~-2.9 to -3.2 |